In [1]:
features_dir = '/Users/kesiyun/Desktop/Thesis/github/features/01SF_sw' 
extra_features_dir = '/Users/kesiyun/Desktop/Thesis/github/features/01SF_sw_aug'
split_path = '/Users/kesiyun/Desktop/Thesis/github/config/00full_video_split_5s.json'
json_path = '/Users/kesiyun/Desktop/Thesis/github/config/00sequenceresult.json'


import os
import json
import re
import glob

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import TensorDataset, random_split

import ast
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, f1_score
import numpy as np


# Utility functions
def read_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def get_training_weight(train_loader):
    def get_loader_stats(loader, name):
        weight_out=[]
        
        label_counts = {}
        feature_shapes = {}
        total_samples = 0

        for batch in loader:
            features, labels = batch
            total_samples += len(labels)

            for feature, label in zip(features, labels):
                label_val = label.item()
                label_counts[label_val] = label_counts.get(label_val, 0) + 1

                shape = tuple(feature.shape)
                feature_shapes[shape] = feature_shapes.get(shape, 0) + 1
        
        for label, count in sorted(label_counts.items()):
            weight_out.append(count)
            #print(f"    Label {label}: {count} samples")
        return weight_out
    return get_loader_stats(train_loader, "Training")    

def print_loader_metadata(train_loader, test_loader):
    def get_loader_stats(loader, name):
        label_counts = {}
        feature_shapes = {}
        total_samples = 0

        for batch in loader:
            features, labels = batch
            total_samples += len(labels)

            for feature, label in zip(features, labels):
                label_val = label.item()
                label_counts[label_val] = label_counts.get(label_val, 0) + 1

                shape = tuple(feature.shape)
                feature_shapes[shape] = feature_shapes.get(shape, 0) + 1

        print(f"\n{name} Loader:")
        print(f"  Total samples: {total_samples}")
        print(f"  Total batches (batch_size={loader.batch_size}): {len(loader)}")
        print(f"  Feature shape(s):")
        for shape, count in feature_shapes.items():
            print(f"    Shape {shape}: {count} samples")
        print(f"  Label distribution:")
        for label, count in sorted(label_counts.items()):
            print(f"    Label {label}: {count} samples")

    get_loader_stats(train_loader, "Training")
    get_loader_stats(test_loader, "Testing")

def get_final_D_cutoff_seconds(meta_for_video):
    """
    Parse the final D(mm:ss) from the 'timesec' string.
    Returns the cutoff in integer seconds, or None if there is no D.
    """
    if not meta_for_video:
        return None
    ts = meta_for_video.get('timesec', '')
    # Find ALL occurrences of D(mm:ss); use the last one as the cutoff.
    matches = list(re.finditer(r'D\((\d+):(\d+)\)', ts))
    if not matches:
        return None
    m = matches[-1]
    minutes = int(m.group(1))
    seconds = int(m.group(2))
    return minutes * 60 + seconds  # total seconds


def parse_augmented_filename(fname):
    """
    Parse names like '99A_0056_X.pt' into (base_id, timestamp_seconds, tag_suffix).
    - If timestamp token is 4 digits, interpret as MMSS.
    - Else if it's an integer string, interpret as total seconds.
    Returns (base_id, timestamp_seconds or None, tag_suffix or None).
    """
    stem = os.path.splitext(os.path.basename(fname))[0]  # e.g., '99A_0056_X'
    parts = stem.split('_')
    base = parts[0] if parts else None
    ts_seconds = None
    tag = parts[-1] if len(parts) >= 3 else (parts[1] if len(parts) == 2 else None)

    if len(parts) >= 2:
        ts_token = parts[1]
        if ts_token.isdigit():
            if len(ts_token) == 4:        # MMSS
                mm = int(ts_token[:2])
                ss = int(ts_token[2:])
                ts_seconds = mm * 60 + ss
            else:                          # total seconds
                ts_seconds = int(ts_token)

    return base, ts_seconds, tag


In [2]:
#prepare the data

#load the data from: 1- 00SF, 2- 01SF_sw

#features_dir = '' 
#split_path = ''
#json_path = ''

loaded_result=read_json(json_path)
dataset_split=read_json(split_path)

training_split=dataset_split['training_set']
test_split=dataset_split['testing_set']
#print(training_split)
#print(test_split)

train_features = []
train_labels = []

for video_name in training_split:
    feature_list_load_1 = torch.load(os.path.join(features_dir, video_name + '.pt'))
    # Determine cutoff (in seconds) from sequenceresult metadata
    cutoff_seconds = get_final_D_cutoff_seconds(loaded_result.get(video_name))

    for i in range(len(feature_list_load_1)):
        feature_this = feature_list_load_1[i][0]  # shape: [1, 2304]
        label_this   = feature_list_load_1[i][1]  # integer 0/1

        if cutoff_seconds is not None:
            # Each entry represents one second; i is 0-based, so second_index = i + 1
            second_index = i + 1
            # After the cutoff second, labels must be 0:
            if second_index > cutoff_seconds:
                label_this = 0

        train_features.append(feature_this)
        train_labels.append(label_this)

    print(video_name, 'added to training (cutoff:', cutoff_seconds, ')')

X_train = torch.cat(train_features, dim=0)              # [num_samples, 2304]
y_train = torch.tensor(train_labels).float().unsqueeze(1)  # [num_samples, 1]

# -------------------------
# Build TEST tensors
# -------------------------
test_features = []
test_labels   = []

for video_name in test_split:
    feature_list_load_1 = torch.load(os.path.join(features_dir, video_name + '.pt'))
    cutoff_seconds = get_final_D_cutoff_seconds(loaded_result.get(video_name))

    for i in range(len(feature_list_load_1)):
        feature_this = feature_list_load_1[i][0]  # shape: [1, 2304]
        label_this   = feature_list_load_1[i][1]  # integer 0/1

        if cutoff_seconds is not None:
            second_index = i + 1
            if second_index > cutoff_seconds:
                label_this = 0

        test_features.append(feature_this)
        test_labels.append(label_this)

    print(video_name, 'added to testing (cutoff:', cutoff_seconds, ')')

X_test = torch.cat(test_features, dim=0)              # [num_samples, 2304]
y_test = torch.tensor(test_labels).float().unsqueeze(1)  # [num_samples, 1]


1A added to training (cutoff: 265 )
1B added to training (cutoff: 245 )
1C added to training (cutoff: 585 )
1D added to training (cutoff: 508 )
1E added to training (cutoff: 215 )
1F added to training (cutoff: 200 )
2A added to training (cutoff: 905 )
3A added to training (cutoff: 178 )
3B added to training (cutoff: None )
3C added to training (cutoff: 305 )
3D added to training (cutoff: 893 )
4B added to training (cutoff: None )
4D added to training (cutoff: 746 )
4F added to training (cutoff: 74 )
5A added to training (cutoff: None )
6A added to training (cutoff: None )
6B added to training (cutoff: 240 )
6C added to training (cutoff: None )
6D added to training (cutoff: 200 )
7A added to training (cutoff: None )
8A added to training (cutoff: None )
8B added to training (cutoff: None )
9A added to training (cutoff: None )
10A added to training (cutoff: None )
10B added to training (cutoff: None )
10C added to training (cutoff: 350 )
11A added to training (cutoff: None )
11B added to 

In [3]:
#aug_

import os
import glob
import torch

# Existing training tensors (already built)
# X_train: [N, 2304]
# y_train: [N, 1]
cutoff_map = {vid: get_final_D_cutoff_seconds(meta) for vid, meta in loaded_result.items()}

#extra_features_dir = ''

extra_features = []
extra_labels = []

# Keep your suffix skip list (using just the single-letter tag before .pt)
skip_tags = {}
#skip_tags = {'W', 'X', 'Y', 'Z'}



for pt_path in glob.glob(os.path.join(extra_features_dir, '*.pt')):
    base_id, ts_seconds, tag = parse_augmented_filename(pt_path)

    # Skip based on tag suffix
    if tag and tag in skip_tags:
        print(os.path.basename(pt_path), f'skipped due to suffix tag {tag}')
        continue

    # Lookup cutoff for the base video
    cutoff_seconds = cutoff_map.get(base_id)

    # If we have both a cutoff and a timestamp, enforce: only include files at or before cutoff
    if cutoff_seconds is not None and ts_seconds is not None and ts_seconds > cutoff_seconds:
        print(os.path.basename(pt_path),
              f'skipped: timestamp {ts_seconds}s exceeds cutoff {cutoff_seconds}s for {base_id}')
        continue

    # Load and append contents
    feature_list = torch.load(pt_path)
    for feat, lab in feature_list:
        # Sanity checks
        assert isinstance(lab, int), f"Label must be int, got {type(lab)} in {pt_path}"
        assert tuple(feat.shape) == (1, 2304), f"Feature shape mismatch in {pt_path}: {feat.shape}"

        extra_features.append(feat)
        extra_labels.append(lab)

    # Informative print
    if cutoff_seconds is None:
        print(os.path.basename(pt_path), f'added to training (augmented) [no cutoff for {base_id}]')
    elif ts_seconds is None:
        print(os.path.basename(pt_path), f'added to training (augmented) [timestamp missing, cutoff {cutoff_seconds}s]')
    else:
        print(os.path.basename(pt_path), f'added to training (augmented) [ts={ts_seconds}s <= cutoff {cutoff_seconds}s]')

# Convert and concatenate to existing training tensors
X_extra = torch.cat(extra_features, dim=0)                              # [N_extra, 2304]
y_extra = torch.tensor(extra_labels, dtype=torch.float32).unsqueeze(1)  # [N_extra, 1]

X_train = torch.cat([X_train, X_extra], dim=0)
y_train = torch.cat([y_train, y_extra], dim=0)


11B_0105_E.pt added to training (augmented) [ts=65s <= cutoff 76s]
1A_0152_X.pt added to training (augmented) [ts=112s <= cutoff 265s]
14D_2007_X.pt added to training (augmented) [ts=1207s <= cutoff 1301s]
13A_1050_F.pt added to training (augmented) [no cutoff for 13A]
11C_0358_D.pt added to training (augmented) [no cutoff for 11C]
12A_1355_W.pt added to training (augmented) [ts=835s <= cutoff 842s]
14E_2041_E.pt added to training (augmented) [ts=1241s <= cutoff 1962s]
17A_0100_H.pt added to training (augmented) [ts=60s <= cutoff 115s]
14E_2459_H.pt added to training (augmented) [ts=1499s <= cutoff 1962s]
10A_0456_E.pt added to training (augmented) [no cutoff for 10A]
15D_0029_X.pt added to training (augmented) [ts=29s <= cutoff 222s]
2A_0014_D.pt added to training (augmented) [ts=14s <= cutoff 905s]
3C_0043_Y.pt added to training (augmented) [ts=43s <= cutoff 305s]
18A_1344_W.pt added to training (augmented) [ts=824s <= cutoff 839s]
14E_2348_Y.pt added to training (augmented) [ts=1428

In [4]:
#Used in Original, weighted loss
# Set a fixed random seed
seed = 3407
generator = torch.Generator().manual_seed(seed)

# ----- Virtual Split from Training Data -----
train_size = int(0.8 * len(X_train))
val_size = len(X_train) - train_size
train_dataset, val_dataset = random_split(
    TensorDataset(X_train, y_train),
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

print_loader_metadata(train_loader, test_loader)

import random
import numpy as np
import torch





Training Loader:
  Total samples: 34927
  Total batches (batch_size=32): 1092
  Feature shape(s):
    Shape (2304,): 34927 samples
  Label distribution:
    Label 0.0: 28169 samples
    Label 1.0: 6758 samples

Testing Loader:
  Total samples: 8172
  Total batches (batch_size=32): 256
  Feature shape(s):
    Shape (2304,): 8172 samples
  Label distribution:
    Label 0.0: 8058 samples
    Label 1.0: 114 samples


In [5]:

# =========================
# BoW-SlowFast (2048/256) classifier with safe metrics (no sklearn warnings)
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, random_split

# -------------------------
# Config & reproducibility
# -------------------------
seed = 3407
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Helpers: convert datasets/loaders to tensors
# -------------------------
def dataset_to_tensors(ds):
    xs, ys = [], []
    for x, y in ds:
        xs.append(x)
        ys.append(y)
    X = torch.stack(xs, dim=0)
    y = torch.stack(ys, dim=0).view(-1, 1)
    return X, y

def loader_to_tensors(loader):
    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb)
        ys.append(yb)
    X = torch.cat(xs, dim=0)
    y = torch.cat(ys, dim=0).view(-1, 1)
    return X, y

# -------------------------
# Slow/Fast split (R50 β⁻¹=8 → 2048 + 256 = 2304)
# -------------------------
def split_slow_fast(X):
    """
    X: (N, 2304) -> (N, 2048), (N, 256)
    """
    assert X.shape[1] == 2304, f"Expected 2304-dim SlowFast embedding, got {X.shape[1]}"
    return X[:, :2048], X[:, 2048:]

# -------------------------
# Mini-batch KMeans in Torch
# -------------------------
class MiniBatchKMeansTorch:
    def __init__(self, n_clusters=128, n_iters=30, batch_size=65536, tol=1e-4, seed=3407):
        self.n_clusters = n_clusters
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.tol = tol
        self.seed = seed
        self.centroids = None

    @torch.no_grad()
    def fit(self, X):
        """
        X: (M, d) tensor (already on device)
        """
        torch.manual_seed(self.seed)
        M, d = X.shape
        K = self.n_clusters

        # init by random sampling
        rand_idx = torch.randperm(M, device=device)[:K]
        C = X[rand_idx].clone()

        for it in range(self.n_iters):
            B = min(self.batch_size, M)
            idx = torch.randperm(M, device=device)[:B]
            batch = X[idx]  # (B, d)

            # assign
            D = torch.cdist(batch, C, p=2)       # (B, K)
            A = D.argmin(dim=1)                  # (B,)

            # recompute centroids from batch
            newC = torch.zeros_like(C)
            counts = torch.zeros(K, device=device, dtype=torch.long)
            for k in range(K):
                mask = (A == k)
                if mask.any():
                    newC[k] = batch[mask].mean(dim=0)
                    counts[k] = mask.sum()

            # keep old where no assignment
            newC[counts == 0] = C[counts == 0]

            shift = (C - newC).pow(2).sum().sqrt().item()
            C = newC
            if shift < self.tol:
                break

        self.centroids = C

    @torch.no_grad()
    def distances(self, X):
        return torch.cdist(X, self.centroids, p=2)  # (N, K)

# -------------------------
# Soft-assignment BoW histograms
# -------------------------
@torch.no_grad()
def soft_bow_hist(X, kmeans: MiniBatchKMeansTorch, soft_k=8, temperature=0.5, normalize='l1'):
    """
    X: (N, d)
    Returns H: (N, K) soft-assignment histograms
    """
    D = kmeans.distances(X)                     # (N, K)
    logits = -D / (temperature + 1e-8)         # closer -> larger logits
    probs = torch.softmax(logits, dim=1)       # (N, K)

    if soft_k is not None and soft_k < probs.shape[1]:
        topk_vals, topk_idx = torch.topk(probs, k=soft_k, dim=1)
        H = torch.zeros_like(probs)
        H.scatter_(1, topk_idx, topk_vals)
    else:
        H = probs

    if normalize == 'l1':
        H = H / (H.sum(dim=1, keepdim=True) + 1e-8)
    elif normalize == 'l2':
        H = H / (torch.norm(H, dim=1, keepdim=True) + 1e-8)
    return H

@torch.no_grad()
def tf_idf(H_train, H_other_list):
    """
    Apply smooth IDF weighting computed on TRAIN histograms.
    """
    N = H_train.shape[0]
    df = (H_train > 0).sum(dim=0)
    idf = torch.log((N + 1) / (df + 1)) + 1.0
    H_train_w = H_train * idf
    H_others_w = [H * idf for H in H_other_list]
    return H_train_w, H_others_w

# -------------------------
# Classifier
# -------------------------
class LinearLogit(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.fc = nn.Linear(D, 1)  # outputs logits
    def forward(self, x):
        return self.fc(x)

# -------------------------
# Safe metrics (warning-free; no sklearn)
# -------------------------
def safe_binary_metrics(preds_tensor, labels_tensor):
    """
    preds_tensor, labels_tensor: tensors in {0,1}, shape (N,)
    Returns: precision, recall, f1, tp, fp, fn
    """
    preds = preds_tensor.view(-1).float()
    y     = labels_tensor.view(-1).float()

    tp = ((preds == 1) & (y == 1)).sum().item()
    fp = ((preds == 1) & (y == 0)).sum().item()
    fn = ((preds == 0) & (y == 1)).sum().item()
    tn = ((preds == 0) & (y == 0)).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1, tp, fp, fn

def f1_at_threshold(logits, labels, thr=0.5):
    probs = torch.sigmoid(logits).view(-1)
    preds = (probs >= thr).float()
    precision, recall, f1, _, _, _ = safe_binary_metrics(preds, labels.view(-1))
    return precision, recall, f1

# -------------------------
# Core training routine with your desired signature
# -------------------------
def train_bow_classifier(
    X_train, y_train, X_test, y_test,
    use_existing_split=False, train_dataset=None, val_dataset=None,
    K_slow=128, K_fast=64, soft_k=8, temperature=0.5, use_tfidf=True,
    lr=1e-3, max_epochs=200, patience=50, plot_filename="loss_curves_bow.png"
):
    """
    If use_existing_split=True, pass train_dataset and val_dataset (from your random_split).
    Otherwise, we'll make an 80/20 validation split from X_train/y_train tensors.
    """

    # -------- prepare split --------
    if use_existing_split and (train_dataset is not None) and (val_dataset is not None):
        X_tr, y_tr = dataset_to_tensors(train_dataset)
        X_va, y_va = dataset_to_tensors(val_dataset)
    else:
        # Create deterministic 80/20 split from given tensors
        generator = torch.Generator().manual_seed(seed)
        full_ds = TensorDataset(X_train, y_train.view(-1, 1))
        train_size = int(0.8 * len(full_ds))
        val_size = len(full_ds) - train_size
        train_dataset, val_dataset = random_split(full_ds, [train_size, val_size], generator=generator)
        X_tr, y_tr = dataset_to_tensors(train_dataset)
        X_va, y_va = dataset_to_tensors(val_dataset)

    # Test tensors (assumed given)
    X_te, y_te = X_test, y_test.view(-1, 1)

    # move to device
    X_tr = X_tr.to(device); y_tr = y_tr.to(device)
    X_va = X_va.to(device); y_va = y_va.to(device)
    X_te = X_te.to(device); y_te = y_te.to(device)

    # -------- Slow/Fast split --------
    Xtr_slow, Xtr_fast = split_slow_fast(X_tr)
    Xva_slow, Xva_fast = split_slow_fast(X_va)
    Xte_slow, Xte_fast = split_slow_fast(X_te)

    # -------- fit KMeans --------
    km_slow = MiniBatchKMeansTorch(n_clusters=K_slow, n_iters=30, batch_size=65536, tol=1e-4, seed=seed)
    km_fast = MiniBatchKMeansTorch(n_clusters=K_fast, n_iters=30, batch_size=65536, tol=1e-4, seed=seed)
    km_slow.fit(Xtr_slow)
    km_fast.fit(Xtr_fast)

    # -------- build soft BoW histograms --------
    Htr_s = soft_bow_hist(Xtr_slow, km_slow, soft_k=soft_k, temperature=temperature, normalize='l1')
    Htr_f = soft_bow_hist(Xtr_fast, km_fast, soft_k=soft_k, temperature=temperature, normalize='l1')
    Hva_s = soft_bow_hist(Xva_slow, km_slow, soft_k=soft_k, temperature=temperature, normalize='l1')
    Hva_f = soft_bow_hist(Xva_fast, km_fast, soft_k=soft_k, temperature=temperature, normalize='l1')
    Hte_s = soft_bow_hist(Xte_slow, km_slow, soft_k=soft_k, temperature=temperature, normalize='l1')
    Hte_f = soft_bow_hist(Xte_fast, km_fast, soft_k=soft_k, temperature=temperature, normalize='l1')

    # optional TF-IDF per stream
    if use_tfidf:
        Htr_s, [Hva_s, Hte_s] = tf_idf(Htr_s, [Hva_s, Hte_s])
        Htr_f, [Hva_f, Hte_f] = tf_idf(Htr_f, [Hva_f, Hte_f])

    # concatenate streams
    Xtr_bow = torch.cat([Htr_s, Htr_f], dim=1)
    Xva_bow = torch.cat([Hva_s, Hva_f], dim=1)
    Xte_bow = torch.cat([Hte_s, Hte_f], dim=1)

    # -------- classifier --------
    model = LinearLogit(Xtr_bow.shape[1]).to(device)
    pos_weight_value = ( (y_tr == 0).sum() / (y_tr == 1).sum() ).item()
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses, val_f1_scores = [], [], []
    best_f1, best_state, best_epoch, no_improve = 0.0, None, 0, 0

    thresholds = torch.linspace(0.05, 0.95, steps=19, device=device)

    for epoch in range(max_epochs):
        # ---- train ----
        model.train()
        optimizer.zero_grad()
        logits_tr = model(Xtr_bow)
        loss_tr = criterion(logits_tr.view(-1), y_tr.view(-1))
        loss_tr.backward()
        optimizer.step()

        train_losses.append(loss_tr.item())

        # ---- validate ----
        model.eval()
        with torch.no_grad():
            logits_va = model(Xva_bow)
            # sweep thresholds for best F1(class 1)
            f1s = []
            for t in thresholds:
                _, _, f1_t = f1_at_threshold(logits_va, y_va, thr=t.item())
                f1s.append(f1_t)
            epoch_best_f1 = max(f1s)
            # also report val loss (threshold-independent)
            loss_va = criterion(logits_va.view(-1), y_va.view(-1)).item()
            val_losses.append(loss_va)
            val_f1_scores.append(epoch_best_f1)

        print(f"Epoch {epoch+1}, Train Loss: {loss_tr.item():.4f}, Val Loss: {loss_va:.4f}, Val F1 (class 1, best thr): {epoch_best_f1:.4f}")

        # early stopping
        if epoch_best_f1 > best_f1 + 1e-6:
            best_f1 = epoch_best_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. Best epoch = {best_epoch+1} (Val F1={best_f1:.4f}).")
                break

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---- choose threshold on validation ----
    model.eval()
    with torch.no_grad():
        logits_va = model(Xva_bow)
        f1s = []
        for t in thresholds:
            _, _, f1_t = f1_at_threshold(logits_va, y_va, thr=t.item())
            f1s.append(f1_t)
        f1s_t = torch.tensor(f1s)
        thr_idx = int(torch.argmax(f1s_t).item())
        best_thr = thresholds[thr_idx].item()

    # ---- plot curves ----
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss', marker='o')
    plt.plot(val_losses, label='Validation Loss', marker='x')
    plt.plot(val_f1_scores, label='Val F1 (class 1)', marker='s')
    plt.axvline(x=best_epoch, color='r', linestyle='--', label='Best Epoch')
    plt.title('BoW-SlowFast Training Progress')
    plt.xlabel('Epoch')
    plt.ylabel('Loss / F1')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(plot_filename)
    plt.close()

    # ---- final test evaluation ----
    with torch.no_grad():
        logits_te = model(Xte_bow)
        probs = torch.sigmoid(logits_te.view(-1))
        preds = (probs >= best_thr).float()

    # safe metrics
    precision_C, recall_C, f1_C, TP_c, FP_c, FN_c = safe_binary_metrics(preds, y_te.view(-1))
    acc = ((preds == y_te.view(-1)).float().mean().item())

    # macro metrics for binary: compute class 0 by flipping
    precision_0, recall_0, f1_0, _, _, _ = safe_binary_metrics(1 - preds, 1 - y_te.view(-1))
    macro_precision = 0.5 * (precision_C + precision_0)
    macro_f1        = 0.5 * (f1_C + f1_0)

    # ----- Output -----
    print("\n--- Detailed Evaluation on Test Set (BoW-SlowFast) ---")
    print(f"Threshold chosen on validation: {best_thr:.2f}")
    print(f"True Positives (C): {TP_c}")
    print(f"False Positives (C): {FP_c}")
    print(f"False Negatives (C): {FN_c}")
    print(f"Precision (C): {precision_C:.4f}")
    print(f"Recall (C): {recall_C:.4f}")
    print(f"F1 Score (C): {f1_C:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_val_f1": best_f1,
        "best_thr": best_thr,
        "Xtr_bow": Xtr_bow, "Xva_bow": Xva_bow, "Xte_bow": Xte_bow
    }

# =========================
# Usage:
# Assuming you already have X_train, y_train, X_test, y_test as torch tensors.
# If you also have train_dataset/val_dataset from your earlier random_split,
# set use_existing_split=True and pass them.
# =========================

# After you created train_dataset, val_dataset above:
# (Uncomment the block below to run)

result = train_bow_classifier(
     X_train, y_train, X_test, y_test,
     use_existing_split=True,           # — tell it to reuse your split
     train_dataset=train_dataset,       # — pass your train dataset
     val_dataset=val_dataset,           # — pass your val dataset
     K_slow=128, K_fast=64, soft_k=8, temperature=0.5, use_tfidf=True,
     lr=1e-3, max_epochs=200, patience=50, plot_filename="loss_curves_bow.png"
)

# Metrics are printed inside train_bow_classifier (same style as your trivial classifier).
# You can also access:
print("Best epoch:", result["best_epoch"])
print("Best Val F1:", result["best_val_f1"])
print("Chosen threshold:", result["best_thr"])


Epoch 1, Train Loss: 1.1199, Val Loss: 1.1147, Val F1 (class 1, best thr): 0.3465
Epoch 2, Train Loss: 1.1171, Val Loss: 1.1120, Val F1 (class 1, best thr): 0.3499
Epoch 3, Train Loss: 1.1145, Val Loss: 1.1092, Val F1 (class 1, best thr): 0.3536
Epoch 4, Train Loss: 1.1118, Val Loss: 1.1066, Val F1 (class 1, best thr): 0.3568
Epoch 5, Train Loss: 1.1091, Val Loss: 1.1039, Val F1 (class 1, best thr): 0.3608
Epoch 6, Train Loss: 1.1064, Val Loss: 1.1012, Val F1 (class 1, best thr): 0.3641
Epoch 7, Train Loss: 1.1038, Val Loss: 1.0985, Val F1 (class 1, best thr): 0.3667
Epoch 8, Train Loss: 1.1012, Val Loss: 1.0959, Val F1 (class 1, best thr): 0.3695
Epoch 9, Train Loss: 1.0985, Val Loss: 1.0932, Val F1 (class 1, best thr): 0.3723
Epoch 10, Train Loss: 1.0959, Val Loss: 1.0906, Val F1 (class 1, best thr): 0.3750
Epoch 11, Train Loss: 1.0933, Val Loss: 1.0880, Val F1 (class 1, best thr): 0.3782
Epoch 12, Train Loss: 1.0907, Val Loss: 1.0854, Val F1 (class 1, best thr): 0.3799
Epoch 13, Tra